# Final evaluation - inference with the v6 model (public score 78)

Inference only. No training, no external data, no network.

**Settings**
- Accelerator: **GPU T4** (x2 is fine but unnecessary - this is one backbone over ~2.4k images)
- Internet: **OFF**. Everything loads from the attached v6 artifacts. This is the
  rule requirement and also the proof of it - Cell 6 reloads the weights with the
  hub disabled and prints a pass/fail.

**Inputs required**
1. The competition dataset (`geo-guessr-final-hackathon-evaluation`)
2. The `geogs_v6` notebook output (contains `artifacts/`)

**Two things this notebook handles that bit us before**

1. The sample file's longitude column is named `pred_long`, but the grader requires
   `pred_lon`. A submission matching the sample exactly gets rejected with
   *"Submission is missing required column(s): ['pred_lon']"*. This notebook writes
   `pred_lon` regardless, and warns if the sample disagrees.
2. `training_dataset/` is mounted alongside `final_test_set/`. A plain recursive
   image glob would sweep in ~19k training images. Test images are taken from
   `final_test_set/` only.

v6 is CLIP-only (2048-dim features, `head_fold*.pt`). The notebook asserts this
rather than assuming it, so a wrong-artifact mixup fails loudly instead of silently
producing garbage.

In [1]:
# =====================================================================
# CELL 1 - Offline mode FIRST (before transformers is imported), then imports
# =====================================================================
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PYTHONHASHSEED"] = "42"

import sys, gc, json, math, time, glob, random, warnings, traceback
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from transformers import CLIPVisionModel

T0 = time.time()
def log(m): print(f"[+{(time.time()-T0)/60:5.1f} min] {m}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

R_EARTH = 6371.0088
IMG, N_VIEWS, BS = 224, 2, 64
DEV = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
log(f"torch {torch.__version__} | device {DEV} | GPUs visible {torch.cuda.device_count()}")

def vec_to_latlon(v):
    v = np.asarray(v, float); v = v/(np.linalg.norm(v, axis=-1, keepdims=True)+1e-12)
    return (np.degrees(np.arcsin(np.clip(v[...,2], -1, 1))),
            np.degrees(np.arctan2(v[...,1], v[...,0])))

def find_all(pat, root="/kaggle/input"):
    return sorted(glob.glob(os.path.join(root, "**", pat), recursive=True))

log("imports OK")

[+  0.0 min] torch 2.10.0+cu128 | device cuda:0 | GPUs visible 2
[+  0.0 min] imports OK


In [2]:
# =====================================================================
# CELL 2 - Locate the v6 artifacts and validate they are internally consistent
# =====================================================================
cands = sorted(set(os.path.dirname(p) for p in find_all("calibration.json")))
log(f"artifact dirs on disk: {cands}")
assert cands, "no calibration.json found - attach the geogs_v6 notebook output"

v6 = [d for d in cands if "v6" in d.lower()]
ART = v6[0] if v6 else cands[0]
log(f">>> USING ARTIFACTS FROM: {ART}")
for d in cands:
    if d != ART: log(f"    (ignoring {d})")

HEADS = sorted(glob.glob(os.path.join(ART, "head_fold*.pt"))) \
        or sorted(glob.glob(os.path.join(ART, "head_seed*.pt")))
assert HEADS, f"no head weights found in {ART}"
log(f"heads: {[os.path.basename(h) for h in HEADS]}")

MU  = np.load(os.path.join(ART, "feat_mu.npy"))
SD  = np.load(os.path.join(ART, "feat_sd.npy"))
CF  = np.load(os.path.join(ART, "fine_centroids.npy"))
CELL_COUNTRY = np.load(os.path.join(ART, "cell_country.npy"))
CAL = json.load(open(os.path.join(ART, "calibration.json")))
log(f"calibration.json -> {CAL}")

LAM   = float(CAL.get("lam", 0.0))
ALPHA = float(CAL.get("alpha", 4.2))
FLOOR = float(CAL.get("floor", 15.0))
R_MAX = float(CAL.get("r_max", 3000.0))
TOP_K = int(CAL.get("topk", 8))
USE_ISO = bool(CAL.get("use_iso", False))
log(f"policy: lambda={LAM} alpha={ALPHA} floor={FLOOR} r_max={R_MAX} top_k={TOP_K}")

if USE_ISO:
    log("!! WARNING: this run selected the ISOTONIC radius family, but the isotonic")
    log("!! model itself was never saved to artifacts - only the flag. Falling back to")
    log("!! the linear alpha policy, which is NOT exactly what produced the 78.")
    log("!! Check the radius distribution in Cell 6 carefully before submitting.")
else:
    log("radius family: linear (alpha x predicted error) - reproducible from artifacts")

sd0 = torch.load(HEADS[0], map_location="cpu")
FEAT_DIM  = sd0["trunk.0.weight"].shape[1]
HID       = sd0["trunk.0.weight"].shape[0]
N_FINE    = sd0["fine.weight"].shape[0]
N_COARSE  = sd0["coarse.weight"].shape[0]
N_COUNTRY = sd0["country.weight"].shape[0]
log(f"checkpoint: feat_dim={FEAT_DIM} hid={HID} fine={N_FINE} "
    f"coarse={N_COARSE} country={N_COUNTRY}")

assert MU.shape[0] == FEAT_DIM, f"feat_mu {MU.shape[0]} != head input {FEAT_DIM}"
assert SD.shape[0] == FEAT_DIM, f"feat_sd {SD.shape[0]} != head input {FEAT_DIM}"
assert CF.shape[0] == N_FINE,   f"centroids {CF.shape[0]} != head fine {N_FINE}"
assert CELL_COUNTRY.shape[0] == N_FINE, "cell_country length != n_fine"
for i, h in enumerate(HEADS):
    assert torch.load(h, map_location="cpu")["trunk.0.weight"].shape == \
           sd0["trunk.0.weight"].shape, f"head {i} disagrees with head 0"
log("consistency checks PASSED")

assert FEAT_DIM == 2048, (f"expected 2048 (CLIP-only v6) but got {FEAT_DIM}. "
                          "A 4096 dim means v7 artifacts were picked up instead.")
CLIP_LOCAL = os.path.join(ART, "clip_vit_l14")
assert os.path.exists(os.path.join(CLIP_LOCAL, "config.json")), \
    f"CLIP weights missing in {ART}"
log("confirmed: CLIP-only model, single backbone")

[+  0.2 min] artifact dirs on disk: ['/kaggle/input/notebooks/mitraasrinivasan1367/geogs-v6/artifacts']
[+  0.2 min] >>> USING ARTIFACTS FROM: /kaggle/input/notebooks/mitraasrinivasan1367/geogs-v6/artifacts
[+  0.2 min] heads: ['head_fold0.pt', 'head_fold1.pt', 'head_fold2.pt', 'head_fold3.pt', 'head_fold4.pt']
[+  0.2 min] calibration.json -> {'lam': 0.25, 'use_iso': False, 'alpha': 4.200000000000001, 'floor': 15.0, 'r_max': 3000.0, 'topk': 8, 'tau': 250.0, 'n_fine': 2000, 'base_score': 0.30638059729088857}
[+  0.2 min] policy: lambda=0.25 alpha=4.200000000000001 floor=15.0 r_max=3000.0 top_k=8
[+  0.2 min] radius family: linear (alpha x predicted error) - reproducible from artifacts
[+  0.2 min] checkpoint: feat_dim=2048 hid=1024 fine=2000 coarse=150 country=298
[+  0.2 min] consistency checks PASSED
[+  0.2 min] confirmed: CLIP-only model, single backbone


In [3]:
# =====================================================================
# CELL 3 - Submission schema and test images
# =====================================================================
ss_all = find_all("sample*submission*.csv")
log(f"sample submission candidates: {ss_all}")
assert ss_all, "no sample submission found - attach the competition dataset"
pref = [p for p in ss_all if "final" in os.path.basename(p).lower()]
SAMPLE_SUB = (pref or ss_all)[0]
if len(ss_all) > 1: log(f">>> chose {SAMPLE_SUB}")
sub_template = pd.read_csv(SAMPLE_SUB)
log(f"sample columns: {list(sub_template.columns)} | rows: {len(sub_template)}")

def pick_col(cols, *k):
    for c in cols:
        lc = c.lower().replace("_","").replace(" ","")
        if all(x in lc for x in k): return c

SRC_ID  = pick_col(sub_template.columns, "id") or sub_template.columns[0]
SRC_LON = pick_col(sub_template.columns, "lon") or pick_col(sub_template.columns, "lng")
assert SRC_LON, "could not find a longitude column in the sample submission"

# The grader requires pred_lon even though the sample file ships pred_long.
OUT_COLS = ["image_id", "pred_lat", "pred_lon", "pred_radius_km"]
if SRC_LON != "pred_lon":
    log(f"NOTE: sample file uses '{SRC_LON}'; the grader requires 'pred_lon'.")
    log("      Writing 'pred_lon' - this is deliberate, not a mismatch.")
log(f"output columns will be: {OUT_COLS}")

test_roots = glob.glob("/kaggle/input/**/final_test_set", recursive=True)
log(f"test roots: {test_roots}")
test_imgs = []
for d in test_roots:
    for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"):
        test_imgs += glob.glob(os.path.join(d, "**", ext), recursive=True)
test_imgs = sorted(set(test_imgs))
log(f"images inside final_test_set: {len(test_imgs)}")

by_stem = {}
for p in test_imgs:
    by_stem.setdefault(os.path.splitext(os.path.basename(p))[0], p)

ids = sub_template[SRC_ID].astype(str).tolist()
TEST_PATHS = [by_stem.get(os.path.splitext(str(t))[0]) for t in ids]
n_missing = sum(p is None for p in TEST_PATHS)

if n_missing:
    log(f"{n_missing} unmatched under final_test_set - widening (excluding training data)")
    wide = []
    for ext in ("*.jpg","*.jpeg","*.png"):
        wide += [p for p in find_all(ext)
                 if "training_dataset" not in p and "noised_dataset" not in p]
    for p in sorted(set(wide)):
        by_stem.setdefault(os.path.splitext(os.path.basename(p))[0], p)
    TEST_PATHS = [by_stem.get(os.path.splitext(str(t))[0]) for t in ids]
    n_missing = sum(p is None for p in TEST_PATHS)

log(f"test images matched: {len(TEST_PATHS)-n_missing}/{len(TEST_PATHS)}")
assert n_missing == 0, f"{n_missing} test images could not be located on disk"

[+  0.3 min] sample submission candidates: ['/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/geo-guessr-final-hackathon-evaluation/sample_submission final.csv']
[+  0.3 min] sample columns: ['image_id', 'pred_lat', 'pred_long', 'pred_radius_km'] | rows: 2448
[+  0.3 min] NOTE: sample file uses 'pred_long'; the grader requires 'pred_lon'.
[+  0.3 min]       Writing 'pred_lon' - this is deliberate, not a mismatch.
[+  0.3 min] output columns will be: ['image_id', 'pred_lat', 'pred_lon', 'pred_radius_km']
[+  0.3 min] test roots: ['/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/geo-guessr-final-hackathon-evaluation/final_test_set']
[+  0.3 min] images inside final_test_set: 2448
[+  0.3 min] test images matched: 2448/2448


In [4]:
# =====================================================================
# CELL 4 - Encode test images with the frozen CLIP backbone.
# View construction is identical to training: full-frame squash + centre square
# crop at 224px. No horizontal flip - mirroring reverses driving side.
# =====================================================================
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

clip = CLIPVisionModel.from_pretrained(CLIP_LOCAL).to(DEV).half().eval()
for p in clip.parameters(): p.requires_grad = False
CDIM = clip.config.hidden_size
MEAN_T = torch.tensor(CLIP_MEAN, device=DEV).view(1,3,1,1).half()
STD_T  = torch.tensor(CLIP_STD,  device=DEV).view(1,3,1,1).half()
log(f"CLIP loaded offline | hidden {CDIM} | {CDIM*N_VIEWS} features per image")
assert CDIM*N_VIEWS == FEAT_DIM, f"backbone gives {CDIM*N_VIEWS}, head wants {FEAT_DIM}"

class ViewDS(Dataset):
    def __init__(s, paths): s.p = [x if x else "" for x in paths]
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        ok = 1
        try: im = Image.open(s.p[i]).convert("RGB")
        except Exception: im = Image.new("RGB",(IMG,IMG),(128,128,128)); ok = 0
        w,h = im.size; q = min(w,h); l,t = (w-q)//2, (h-q)//2
        a = im.resize((IMG,IMG), Image.BICUBIC)
        b = im.crop((l,t,l+q,t+q)).resize((IMG,IMG), Image.BICUBIC)
        return (torch.stack([
            torch.from_numpy(np.asarray(a, dtype=np.uint8)).permute(2,0,1),
            torch.from_numpy(np.asarray(b, dtype=np.uint8)).permute(2,0,1)]), ok)

@torch.no_grad()
def encode(paths):
    dl = DataLoader(ViewDS(paths), batch_size=BS, shuffle=False,
                    num_workers=4, pin_memory=True)
    out, bad, seen = [], 0, 0
    for x, ok in dl:
        n = x.shape[0]
        xx = x.to(DEV, non_blocking=True).reshape(n*N_VIEWS,3,IMG,IMG).half().div_(255.)
        f = clip(pixel_values=(xx-MEAN_T)/STD_T).pooler_output
        out.append(f.reshape(n, N_VIEWS*CDIM).float().cpu().numpy())
        bad += int((ok==0).sum()); seen += n
        if seen % (BS*10) < BS: log(f"  encoded {seen}/{len(paths)}")
    return np.concatenate(out), bad

log(f"encoding {len(TEST_PATHS)} test images ...")
Z, n_bad = encode(TEST_PATHS)
log(f"features {Z.shape} | unreadable images: {n_bad}")
assert Z.shape[1] == FEAT_DIM
clip.cpu(); del clip; gc.collect(); torch.cuda.empty_cache()
Xt = torch.tensor((Z-MU)/SD, dtype=torch.float32, device=DEV)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[+  0.6 min] CLIP loaded offline | hidden 1024 | 2048 features per image
[+  0.6 min] encoding 2448 test images ...
[+  0.8 min]   encoded 640/2448
[+  1.0 min]   encoded 1280/2448
[+  1.2 min]   encoded 1920/2448
[+  1.3 min] features (2448, 2048) | unreadable images: 0


In [5]:
# =====================================================================
# CELL 5 - Rebuild heads, ensemble, decode to coordinate + radius
# =====================================================================
CFT = torch.tensor(CF, dtype=torch.float32, device=DEV)
CCT = torch.tensor(np.where(CELL_COUNTRY>=0, CELL_COUNTRY, 0), device=DEV)
CC_VALID = torch.tensor((CELL_COUNTRY>=0).astype(np.float32), device=DEV)

class GeoHead(nn.Module):
    def __init__(s, d, nf, nc, nk, hid):
        super().__init__()
        s.trunk = nn.Sequential(
            nn.Linear(d,hid), nn.LayerNorm(hid), nn.GELU(), nn.Dropout(0.30),
            nn.Linear(hid,hid), nn.LayerNorm(hid), nn.GELU(), nn.Dropout(0.15))
        s.fine=nn.Linear(hid,nf); s.coarse=nn.Linear(hid,nc); s.country=nn.Linear(hid,nk)
        s.delta=nn.Linear(hid,3); s.unc=nn.Linear(hid,1)
    def forward(s,x):
        h=s.trunk(x)
        return s.fine(h), s.coarse(h), s.country(h), s.delta(h), s.unc(h).squeeze(-1)

def decode_point(fl, delta, topk=TOP_K, max_spread_km=600.0):
    """Weighted top-K centroid, restricted to cells near the top-1 cell.
    Blending cells on opposite continents lands the point in open ocean."""
    p = torch.softmax(fl.float(), 1)
    w, idx = torch.topk(p, topk, 1)
    anchor = CFT[idx[:,0]]; cand = CFT[idx]
    cos = (cand*anchor.unsqueeze(1)).sum(-1).clamp(-1+1e-9, 1-1e-9)
    w = w * ((R_EARTH*torch.acos(cos)) <= max_spread_km).float()
    w = w / (w.sum(1, keepdim=True) + 1e-9)
    base = (cand*w.unsqueeze(-1)).sum(1)
    base = base / (base.norm(dim=1, keepdim=True) + 1e-9)
    v = base + 0.08*torch.tanh(delta)
    return v / (v.norm(dim=1, keepdim=True) + 1e-9)

def country_adjust(fl, cl, lam):
    if lam <= 0: return fl
    lp = torch.log_softmax(fl.float(), 1); pc = torch.softmax(cl.float(), 1)
    return lp + lam*torch.log(pc[:,CCT]*CC_VALID.unsqueeze(0) + 1e-6)

Tf = torch.zeros(len(Xt), N_FINE,    device=DEV)
Tc = torch.zeros(len(Xt), N_COUNTRY, device=DEV)
Td = torch.zeros(len(Xt), 3,         device=DEV)
Tu = torch.zeros(len(Xt),            device=DEV)
with torch.no_grad():
    for hp in HEADS:
        m = GeoHead(FEAT_DIM, N_FINE, N_COARSE, N_COUNTRY, HID).to(DEV)
        m.load_state_dict(torch.load(hp, map_location=DEV)); m.eval()
        for i in range(0, len(Xt), 4096):
            fl,cl,ctl,dl_,ul = m(Xt[i:i+4096])
            Tf[i:i+4096] += torch.softmax(fl.float(),1)
            Tc[i:i+4096] += torch.softmax(ctl.float(),1)
            Td[i:i+4096] += dl_.float()
            Tu[i:i+4096] += ul.float()
        del m; log(f"  ensembled {os.path.basename(hp)}")
K = len(HEADS); Tf/=K; Tc/=K; Td/=K; Tu/=K

v = decode_point(country_adjust(torch.log(Tf+1e-12), torch.log(Tc+1e-12), LAM), Td)
p_lat, p_lon = vec_to_latlon(v.cpu().numpy())
p_rad = np.clip(ALPHA*np.maximum(np.expm1(Tu.cpu().numpy()), 1.0), FLOOR, R_MAX)
log(f"decoded | radius min {p_rad.min():.0f} median {np.median(p_rad):.0f} max {p_rad.max():.0f} km")

[+  1.4 min]   ensembled head_fold0.pt
[+  1.4 min]   ensembled head_fold1.pt
[+  1.4 min]   ensembled head_fold2.pt
[+  1.4 min]   ensembled head_fold3.pt
[+  1.4 min]   ensembled head_fold4.pt
[+  1.4 min] decoded | radius min 237 median 596 max 1221 km


In [6]:
# =====================================================================
# CELL 6 - Ocean rescue, write submission, verify offline compliance
# =====================================================================
geoms, tree = [], None
try:
    import shapely
    from shapely.geometry import shape, Point
    from shapely.strtree import STRtree
    from shapely.ops import nearest_points
    gj = find_all("*.geojson")
    log(f"geojson files: {gj}")
    if gj:
        for ft_ in json.load(open(gj[0], encoding="utf-8"))["features"]:
            try:
                g = shape(ft_["geometry"]); geoms.append(g if g.is_valid else g.buffer(0))
            except Exception: pass
    if geoms:
        tree = STRtree(geoms)
        pts = shapely.points(p_lon.astype(float), p_lat.astype(float))
        inside = np.zeros(len(pts), bool)
        pr = tree.query(pts, predicate="intersects"); inside[pr[0]] = True
        ocean = np.where(~inside)[0]
        log(f"points outside every country: {len(ocean)}/{len(pts)}")
        for j in ocean:
            try:
                pt = Point(float(p_lon[j]), float(p_lat[j]))
                gi = tree.nearest(pt)
                gi = int(gi if np.isscalar(gi) else np.asarray(gi).ravel()[0])
                q,_ = nearest_points(geoms[gi], pt); p_lat[j], p_lon[j] = q.y, q.x
            except Exception: pass
        log("snapped offshore points to nearest land")
except Exception as e:
    log(f"ocean snap skipped ({e})")

p_lat = np.clip(p_lat, -90, 90)
p_lon = ((p_lon + 180) % 360) - 180
bad = ~np.isfinite(p_lat) | ~np.isfinite(p_lon) | ~np.isfinite(p_rad)
if bad.any():
    log(f"WARNING {int(bad.sum())} non-finite -> safe defaults")
    p_lat[bad], p_lon[bad], p_rad[bad] = 0.0, 0.0, 2000.0

sub = pd.DataFrame({
    "image_id":       sub_template[SRC_ID].astype(str).values,
    "pred_lat":       p_lat,
    "pred_lon":       p_lon,
    "pred_radius_km": p_rad,
})[OUT_COLS]
OUT = "/kaggle/working/submission.csv"
sub.to_csv(OUT, index=False)
log(f"WROTE {OUT}")
print(sub.head(8).to_string())

log("")
log("--- offline compliance ---")
try:
    _ = CLIPVisionModel.from_pretrained(CLIP_LOCAL); del _
    log("CLIP reloads from local path with HF_HUB_OFFLINE=1  [PASS]")
    h = GeoHead(FEAT_DIM, N_FINE, N_COARSE, N_COUNTRY, HID)
    h.load_state_dict(torch.load(HEADS[0], map_location="cpu")); del h
    log("head reloads from artifacts                        [PASS]")
    log(">>> no network used anywhere in the inference path <<<")
except Exception as e:
    log(f"offline check FAILED: {e}")

[+  1.5 min] geojson files: ['/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/country_boundaries.geojson', '/kaggle/input/datasets/mitraasrinivasan/geo-guessr-final-hackathon-evaluation/geo-guessr-final-hackathon-evaluation/country_boundaries.geojson']
[+  1.5 min] points outside every country: 287/2448
[+  1.5 min] snapped offshore points to nearest land
[+  1.5 min] WROTE /kaggle/working/submission.csv
               image_id   pred_lat    pred_lon  pred_radius_km
0  a88766ed092cd328.jpg  53.767291   19.876442      564.414795
1  1d1b147ac111029f.jpg   1.030991  104.003979      279.244720
2  8703a028b9392b48.jpg  56.948556   23.419861      593.052979
3  4366f03b294ff3d7.jpg  16.271927  -61.695191      743.617920
4  251dd7fa406aeaa1.jpg  59.617489   16.675710      555.016052
5  15c3c59ced2f82fe.jpg  61.519159   16.821400      600.119141
6  f4f50202fe33f2b8.jpg  44.039434   21.066952      490.050110
7  ea05b40b6df1e33e.jpg  61.507603   24.264848      735.88

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[+  1.5 min] CLIP reloads from local path with HF_HUB_OFFLINE=1  [PASS]
[+  1.5 min] head reloads from artifacts                        [PASS]
[+  1.5 min] >>> no network used anywhere in the inference path <<<


In [7]:
# =====================================================================
# CELL 7 - PRE-SUBMISSION CHECKS. Read every line before uploading.
# =====================================================================
ok_all = True
def check(name, cond, detail=""):
    global ok_all
    ok_all &= bool(cond)
    print(f"  [{'PASS' if cond else 'FAIL'}] {name} {detail}", flush=True)

print("="*74)
check("row count matches sample", len(sub)==len(sub_template),
      f"({len(sub)} vs {len(sub_template)})")
check("columns are exactly the grader's schema", list(sub.columns)==OUT_COLS,
      f"{list(sub.columns)}")
check("'pred_lon' present (grader requirement)", "pred_lon" in sub.columns)
check("image_id order preserved",
      (sub["image_id"].values == sub_template[SRC_ID].astype(str).values).all())
check("no NaNs", sub.isna().sum().sum()==0)
check("no duplicate ids", sub["image_id"].duplicated().sum()==0)
check("lat within [-90,90]", sub["pred_lat"].between(-90,90).all())
check("lon within [-180,180]", sub["pred_lon"].between(-180,180).all())
check("radius > 0", (sub["pred_radius_km"]>0).all())

r = sub["pred_radius_km"].values
n_cap = int((r >= R_MAX-1).sum()); n_uniq = len(np.unique(np.round(r,1)))
check("radii NOT all pinned at cap", n_cap < 0.5*len(r), f"({n_cap}/{len(r)} at {R_MAX:.0f})")
check("radii varied", n_uniq > 0.3*len(r), f"({n_uniq} distinct)")
print(f"  radius  min {r.min():7.0f} | median {np.median(r):7.0f} | max {r.max():7.0f} km")

lat_u = len(np.unique(np.round(sub["pred_lat"].values,3)))
check("coordinates varied", lat_u > 0.3*len(sub), f"({lat_u} distinct latitudes)")

if tree is not None:
    pts2 = shapely.points(sub["pred_lon"].values.astype(float),
                          sub["pred_lat"].values.astype(float))
    ins = np.zeros(len(pts2), bool)
    pr2 = tree.query(pts2, predicate="intersects"); ins[pr2[0]] = True
    check("few points left in open ocean", int((~ins).sum()) < 0.10*len(sub),
          f"({int((~ins).sum())}/{len(sub)})")

print("="*74)
print("ALL CHECKS PASSED - safe to submit" if ok_all
      else "SOMETHING FAILED - do NOT submit until you understand why")
print("="*74)
print(f"artifacts : {ART}")
print(f"model     : CLIP-only, {len(HEADS)} heads, hidden {HID}")
print(f"policy    : lambda={LAM}, alpha={ALPHA}, floor={FLOOR}")
print(f"file      : {OUT}")

  [PASS] row count matches sample (2448 vs 2448)
  [PASS] columns are exactly the grader's schema ['image_id', 'pred_lat', 'pred_lon', 'pred_radius_km']
  [PASS] 'pred_lon' present (grader requirement) 
  [PASS] image_id order preserved 
  [PASS] no NaNs 
  [PASS] no duplicate ids 
  [PASS] lat within [-90,90] 
  [PASS] lon within [-180,180] 
  [PASS] radius > 0 
  [PASS] radii NOT all pinned at cap (0/2448 at 3000)
  [PASS] radii varied (1866 distinct)
  radius  min     237 | median     596 | max    1221 km
  [PASS] coordinates varied (2347 distinct latitudes)
  [PASS] few points left in open ocean (130/2448)
ALL CHECKS PASSED - safe to submit
artifacts : /kaggle/input/notebooks/mitraasrinivasan1367/geogs-v6/artifacts
model     : CLIP-only, 5 heads, hidden 1024
policy    : lambda=0.25, alpha=4.200000000000001, floor=15.0
file      : /kaggle/working/submission.csv
